In [2]:
import numpy as np
from scipy.stats import norm

def calculate_precise_var():
    # ==========================================
    # 1. 输入参数 (全部使用浮点数以保证精度)
    # ==========================================
    # 资产 A
    shares_a = -10000.0
    price_a = 50.0
    vol_a = 0.35
    beta_a = 0.250

    # 资产 B
    shares_b = 10000.0
    price_b = 100.0
    vol_b = 0.45
    beta_b = 1.500

    # 市场参数
    vol_m = 0.30
    annualization_factor = 250.0
    confidence_level = 0.95

    # ==========================================
    # 2. 基础组合计算 (不进行任何手动四舍五入)
    # ==========================================
    val_a = shares_a * price_a
    val_b = shares_b * price_b
    val_p = val_a + val_b  # 组合总价值 (500,000)

    w_a = val_a / val_p    # -1.0
    w_b = val_b / val_p    #  2.0

    # ==========================================
    # 3. 方差分解 (Variance Decomposition)
    # ==========================================
    # 逻辑：总方差 = 系统性方差 + 特质方差
    # 特质方差 (Sigma_epsilon^2) = Sigma_total^2 - (Beta * Sigma_m)^2

    # A 的特质方差
    var_idio_a = vol_a**2 - (beta_a * vol_m)**2
    
    # B 的特质方差
    var_idio_b = vol_b**2 - (beta_b * vol_m)**2

    # ==========================================
    # 4. 组合方差计算
    # ==========================================
    
    # --- 组合的系统性风险 ---
    beta_p = w_a * beta_a + w_b * beta_b
    var_sys_p = (beta_p * vol_m)**2

    # --- 组合的特质风险 ---
    # 假设特质收益不相关，方差具有可加性：w_a^2 * var_a + w_b^2 * var_b
    var_idio_p = (w_a**2 * var_idio_a) + (w_b**2 * var_idio_b)

    # --- 总方差 (对冲前) ---
    var_total_p = var_sys_p + var_idio_p

    # --- 年化波动率 ---
    vol_annual_before = np.sqrt(var_total_p)  # 对冲前 (含 Beta)
    vol_annual_after = np.sqrt(var_idio_p)    # 对冲后 (不含 Beta)

    # ==========================================
    # 5. VaR 计算
    # ==========================================
    # 使用 scipy 计算最精确的 Z-score，不手动截断
    z_score = norm.ppf(confidence_level)
    
    # 调整时间因子 sqrt(1/250)
    time_factor = 1.0 / np.sqrt(annualization_factor)

    # 计算 VaR
    # Before: 基于总波动率
    var_before = val_p * vol_annual_before * time_factor * z_score
    
    # After: 基于特质波动率 (系统性风险已被期货对冲移除)
    var_after = val_p * vol_annual_after * time_factor * z_score

    # ==========================================
    # 6. 结果输出 (仅在此处进行 4 位小数格式化)
    # ==========================================
    print(f"{'='*40}")
    print(f"高精度运算结果核对")
    print(f"{'='*40}")
    
    print(f"Z-score (未截断): {z_score}")
    print(f"组合 Beta: {beta_p}")
    print(f"组合总价值: {val_p}")
    print(f"{'-'*40}")
    
    print(f"中间变量 (年化波动率，未截断):")
    print(f"  对冲前波动率: {vol_annual_before}")
    print(f"  对冲后波动率: {vol_annual_after}")
    print(f"{'-'*40}")
    
    print(f"最终结果 (保留 4 位小数):")
    print(f"1. VaR Before Hedging: {var_before:.4f}")
    print(f"2. VaR After Hedging:  {var_after:.4f}")
    print(f"{'='*40}")

if __name__ == "__main__":
    calculate_precise_var()

高精度运算结果核对
Z-score (未截断): 1.6448536269514722
组合 Beta: 2.75
组合总价值: 500000.0
----------------------------------------
中间变量 (年化波动率，未截断):
  对冲前波动率: 0.8930285549745877
  对冲后波动率: 0.3418698582794339
----------------------------------------
最终结果 (保留 4 位小数):
1. VaR Before Hedging: 46450.7363
2. VaR After Hedging:  17782.3056
